# Movie identification with BERG in silico EEG (Option A)

**New approach, same dataset.** Instead of training our own encoding model
(AlexNet features -> PCA -> RidgeCV -> EEG, as in notebook 01/02), we use the
**Brain Encoding Response Generator (BERG)** - a toolbox of *pretrained*
encoding models that produce *in silico* (simulated) neural responses to
arbitrary stimuli.

Pipeline (no training on our side):
1. Load a pretrained BERG EEG encoding model.
2. For each test video, take its frame(s) and ask BERG to generate an in
   silico EEG response.
3. Average the per-frame in silico responses into one response vector per video.
4. Match those simulated responses against our **real measured** test EEG using
   correlation, and rank the candidate videos for each measured recording.
5. Score top-1 / top-5 against the ground truth.

> Note on shapes: the BERG model `eeg-things_eeg_2-vit_b_32` was trained on the
> THINGS-EEG2 dataset (image stimuli, its own channel/time layout, e.g. 63x250),
> which does **not** match our 17x300 movie EEG. We therefore compare in a
> representational way: each response is flattened and z-scored, and matching is
> done with correlation on a common feature length. This tolerates the layout
> mismatch but means absolute values are not directly comparable to notebook 02.

## Requirements
- Python >= 3.11 (BERG requirement).
- `pip install -U git+https://github.com/gifale95/BERG.git`
- Download the BERG data/model weights (AWS CLI, no account needed):
  `aws s3 sync --no-sign-request s3://brain-encoding-response-generator ./brain-encoding-response-generator`
  (or just the EEG model subfolder). Set `BERG_DIR` below to that folder.

In [15]:
# --- Optional one-time setup (uncomment to run inside the notebook) ---
# %pip install -U "git+https://github.com/gifale95/BERG.git"
#
# Download only the EEG encoding model weights (smaller than the full bucket):
# !aws s3 sync --no-sign-request \
#   s3://brain-encoding-response-generator/encoding_models/modality-eeg \
#   ./brain-encoding-response-generator/encoding_models/modality-eeg

In [16]:
# --- Windows UTF-8 fix (MUST run before `import berg`) ---
# BERG opens its YAML model cards with open(path, 'r') and no encoding=, so on
# Windows Python defaults to cp1252 and crashes on UTF-8 bytes (UnicodeDecodeError
# 'charmap' ... 0x9d). We make text-mode open() default to UTF-8 for this kernel.
import builtins
if not getattr(builtins, '_utf8_open_patched', False):
    _orig_open = builtins.open
    def _utf8_open(*args, **kwargs):
        # mode may be positional (args[1]) or a keyword; binary mode -> leave alone
        mode = kwargs.get('mode', args[1] if len(args) > 1 else 'r')
        if 'b' not in mode and kwargs.get('encoding') is None:
            kwargs['encoding'] = 'utf-8'
        return _orig_open(*args, **kwargs)
    builtins.open = _utf8_open
    builtins._utf8_open_patched = True
    print('patched open() to default encoding=utf-8')

import os, glob
import numpy as np
import cv2
from scipy.signal import decimate

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

device: cuda


In [17]:
# ---------------- Config ----------------
SUB        = 1

# Path to the downloaded BERG toolkit folder (see setup cell above)
BERG_DIR   = './brain-encoding-response-generator'
MODEL_ID   = 'eeg-things_eeg_2-vit_b_32'   # pretrained EEG encoding model

# Data folders (same convention as notebooks 01/02)
TEST_DIR         = './tst'
STIM_TEST_DIR    = './stimuli_test'
FRAME_TEST_DIR   = './frames_test'
GT_DIR           = './tst_stim'

# EEG preprocessing (must match how the measured EEG was handled before)
SFREQ      = 500
DEC        = 5                 # 500 Hz -> 100 Hz
N_FRAMES   = 8                 # frames sampled per video

# Test stimulus ids (videos 1001..1102)
tst_ids = list(range(1001, 1103))
print(len(tst_ids), 'test ids')

102 test ids


## 1. Load the real measured test EEG (this is the ground-truth signal we match against)
Same decimation as notebooks 01/02 so the measured side is treated consistently.

In [18]:
tst_eeg = np.load(f'{TEST_DIR}/sub-{SUB:02d}_tst.npy')   # (102, 17, 1500)
print('raw measured EEG:', tst_eeg.shape)

tst_eeg = decimate(tst_eeg, DEC, axis=-1, ftype='fir', zero_phase=True)
n_stim, n_chan, n_time = tst_eeg.shape
print('decimated measured EEG:', tst_eeg.shape)

raw measured EEG: (102, 17, 1500)
decimated measured EEG: (102, 17, 300)


## 2. Collect one or more frames per test video
We reuse the same frame-dumping logic as before. The matcher compares file
basenames so it is Windows/Unix safe (this was a bug in notebook 02).

In [19]:
vids = sorted(glob.glob(os.path.join(STIM_TEST_DIR, '*.mp4')))
print(len(vids), 'videos found')

def video_for(stim_id):
    target = f'{int(stim_id)}.mp4'
    hits = [v for v in vids if os.path.basename(v) == target]
    assert hits, f'no video found for stimulus {stim_id} (expected {target})'
    return hits[0]

def dump_frames(stim_ids, out_dir, n_frames=N_FRAMES):
    os.makedirs(out_dir, exist_ok=True)
    for sid in stim_ids:
        cap = cv2.VideoCapture(video_for(sid))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        picks = np.linspace(0, max(total - 1, 0), n_frames).astype(int)
        for k, fno in enumerate(picks):
            path = os.path.join(out_dir, f'{int(sid):05d}_f{k:02d}.jpg')
            if os.path.exists(path):
                continue
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(fno))
            ok, frame = cap.read()
            if ok:
                cv2.imwrite(path, frame)
        cap.release()

dump_frames(tst_ids, FRAME_TEST_DIR)
print(len(glob.glob(f'{FRAME_TEST_DIR}/*.jpg')), 'frames available')

102 videos found
816 frames available


In [20]:
def frames_for(sid, out_dir=FRAME_TEST_DIR):
    """Return the list of frame file paths for one stimulus id, sorted."""
    return sorted(glob.glob(os.path.join(out_dir, f'{int(sid):05d}_f*.jpg')))

def load_rgb(path):
    """Load an image as RGB uint8 HxWx3 (BERG/torchvision expect RGB)."""
    bgr = cv2.imread(path)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

# sanity check
_f = frames_for(tst_ids[0])
print(tst_ids[0], '->', len(_f), 'frames; example:', _f[:1])

1001 -> 8 frames; example: ['./frames_test\\01001_f00.jpg']


## 3. Load the pretrained BERG EEG encoding model

In [21]:
# Self-contained UTF-8 guard: runs right here so it cannot be out of order
# relative to the berg import (the earlier patch cell may not have run, or
# berg may have been imported before it). Safe to run repeatedly.
import builtins, sys
if not getattr(builtins, '_utf8_open_patched', False):
    _orig_open = builtins.open
    def _utf8_open(*a, **kw):
        mode = kw.get('mode', a[1] if len(a) > 1 else 'r')
        if 'b' not in mode and kw.get('encoding') is None:
            kw['encoding'] = 'utf-8'
        return _orig_open(*a, **kw)
    builtins.open = _utf8_open
    builtins._utf8_open_patched = True

# If berg was already imported (before the patch), drop it so the next import
# re-reads its YAML model cards with the patched, UTF-8 open().
for _m in [m for m in list(sys.modules) if m == 'berg' or m.startswith('berg.')]:
    del sys.modules[_m]

from berg import BERG

berg = BERG(berg_dir=BERG_DIR)

eeg_model = berg.get_encoding_model(MODEL_ID, subject=1, device=DEVICE)

# Inspect metadata so we know the in silico output layout (channels/time, etc.)
try:
    meta = berg.get_model_metadata(MODEL_ID, subject=1)
    print('metadata keys:', list(meta.keys()) if hasattr(meta, 'keys') else type(meta))
except Exception as e:
    print('metadata unavailable:', e)

Model loaded on cuda for subject 1
metadata keys: ['eeg', 'encoding_models']


## 4. Generate in silico EEG for every test video
For each video we encode its frames, then average the resulting in silico
responses into one vector per video. We discover the output shape at runtime
from the first successful `encode` call instead of hardcoding it.

In [22]:
# BERG's generate_response docstring requires the stimulus to be a 4-D array
# (Batch, 3 RGB channels, Width, Height), SQUARE, with integer values [0, 255].
# So we resize each frame to a square and move channels to the front.
IMG_SIZE = 224   # square side fed to the ViT-B/32 preprocessing

def to_berg_stimulus(rgb_images):
    """list of HxWx3 uint8 RGB -> (N, 3, IMG_SIZE, IMG_SIZE) uint8 array."""
    out = []
    for im in rgb_images:
        sq = cv2.resize(im, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        out.append(np.transpose(sq, (2, 0, 1)))   # HWC -> CHW
    return np.stack(out).astype('uint8')           # (N, 3, H, W)

def encode_images(images):
    """Run BERG on a list of RGB images -> in silico responses.
    BERG returns (batch, 4 repetitions, n_channels, n_times); we average over
    the repetition axis so each image yields one (n_channels, n_times) response.
    """
    stim = to_berg_stimulus(images)
    out = np.asarray(berg.encode(eeg_model, stim))   # (N, 4, C, T)
    if out.ndim == 4:
        out = out.mean(axis=1)                        # average the 4 repetitions -> (N, C, T)
    return out

# One quick probe to learn the output shape
_probe = encode_images([load_rgb(frames_for(tst_ids[0])[0])])
print('in silico output shape for 1 image:', _probe.shape)

Encoding EEG responses: 100%|██████████| 1/1 [00:00<00:00,  1.05it/s, Encoded images=1, Total images=1]

in silico output shape for 1 image: (1, 63, 140)


In [23]:
insilico_rows = []
for sid in tst_ids:
    paths = frames_for(sid)
    assert paths, f'no frames for stimulus {sid}'
    imgs = [load_rgb(p) for p in paths]
    resp = encode_images(imgs)              # (n_frames, <model layout>)
    resp = resp.reshape(resp.shape[0], -1)  # flatten each frame response
    insilico_rows.append(resp.mean(axis=0)) # average across the video's frames

X_silico = np.stack(insilico_rows)          # (102, D_silico)
print('in silico responses per video:', X_silico.shape)

Encoding EEG responses: 100%|██████████| 1/1 [00:00<00:00,  2.83it/s, Encoded images=8, Total images=8]

in silico responses per video: (102, 8820)


## 5. Match in silico responses against the real measured EEG
The two sides live in different spaces (BERG's THINGS-EEG2 layout vs our
17x300 movie EEG), so we can't correlate them element-by-element. We build a
z-scored feature vector from each side and use a **cross-correlation of the
sorted/aligned feature profiles** via a common projection: reduce both sides to
the same dimensionality with PCA fit on each side independently, then match by
correlation. This is a representational-similarity style match.

> This is the pragmatic way to compare two differently-shaped response spaces.
> It will not beat a model trained directly on the subject, but it tests how
> much a *pretrained, untuned* in silico model already aligns with real brain
> responses to these movies.

In [24]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Measured EEG: use the same informative window as notebook 02 ([5:60]) then flatten
M = tst_eeg[:, :, 5:60].reshape(len(tst_ids), -1)          # (102, 17*55)

# Reduce both sides to a shared number of components
k = min(50, M.shape[0] - 1, M.shape[1], X_silico.shape[1])

M_p = PCA(n_components=k).fit_transform(StandardScaler().fit_transform(M))
S_p = PCA(n_components=k).fit_transform(StandardScaler().fit_transform(X_silico))

def zrows(A):
    return (A - A.mean(1, keepdims=True)) / (A.std(1, keepdims=True) + 1e-8)

Mz = zrows(M_p)
Sz = zrows(S_p)

# Similarity between each measured recording (row) and each in silico video (col)
Sim = Mz @ Sz.T / Mz.shape[1]              # (102 measured, 102 videos)
print('similarity matrix:', Sim.shape)

similarity matrix: (102, 102)


## 6. Ranked candidate list + scoring
`gt` gives the true video id for each measured recording. We align the measured
rows to the stimulus order the in silico columns use (columns j -> video 1001+j).

In [28]:
video_ids = np.array(tst_ids)                 # column j -> video_ids[j]
order_idx = np.argsort(-Sim, axis=1)          # best-first column indices per row
ranked    = video_ids[order_idx]              # (102, 102) video ids, best first
print('ranked candidate list:', ranked.shape)
ranked

ranked candidate list: (102, 102)


array([[1089, 1056, 1051, ..., 1054, 1029, 1046],
       [1009, 1090, 1033, ..., 1074, 1002, 1059],
       [1045, 1040, 1051, ..., 1058, 1059, 1096],
       ...,
       [1089, 1033, 1067, ..., 1084, 1063, 1065],
       [1013, 1093, 1076, ..., 1006, 1008, 1065],
       [1029, 1006, 1008, ..., 1072, 1038, 1005]], shape=(102, 102))

In [26]:
gt_path = f'{GT_DIR}/sub-{SUB:02d}_tst_stim.npy'
if os.path.exists(gt_path):
    gt = np.load(gt_path)                     # true video id per measured row
    truth = gt                                # column order already matches tst_ids
    top1 = (ranked[:, 0] == truth).mean()
    top5 = (ranked[:, :5] == truth[:, None]).any(axis=1).mean()
    print(f'top-1 {top1:.4f}  (chance {1/len(tst_ids):.4f})')
    print(f'top-5 {top5:.4f}  (chance {5/len(tst_ids):.4f})')
else:
    print('no ground truth for this subject; skipping scoring.')

top-1 0.0196  (chance 0.0098)
top-5 0.0784  (chance 0.0490)


In [27]:
np.save(f'sub-{SUB:02d}_ranked_candidate_list_berg.npy', ranked)
print('saved sub-{:02d}_ranked_candidate_list_berg.npy'.format(SUB))

saved sub-01_ranked_candidate_list_berg.npy


## Notes & next steps
- `eeg-things_eeg_2-vit_b_32` is trained on **image** viewing (THINGS-EEG2), not
  movies, so it is an out-of-distribution test. Expect modest numbers; the point
  is to measure how far a *pretrained* in silico model transfers with zero tuning.
- If results are weak, Option B (feed BERG responses as features into a RidgeCV
  trained on this subject's real training EEG) will usually do better because it
  learns the subject-specific mapping.
- BERG also exposes **video** encoding models (e.g. `fmri-bmd-s3d`,
  `calcium_2p-wang_2025-3DCNN`) - but those are fMRI/calcium, not EEG. For an
  EEG + video match you would need a video-native EEG encoding model.
- To run for other subjects, change `SUB` and re-run. Ground truth only exists
  for subjects 01-04 (`tst_stim/`), so 05/06 will produce a ranked list but no score.